# Task 11 -- Harmonic climatology quasi-stationarity (`docs/open_questions.md` §7.1)

Refits `train.climatology.ClimatologyRemover` over multiple overlapping
windows of the same variable and compares the recovered trend coefficient
`c_lin_` map -- directly testing whether a reported trend is a
window-independent physical constant or an artifact of the specific window
used. See `tasks/11-climatology-quasi-stationarity.md`.

**Note:** `sst` is the variable this concern is framed around
(`harmonic_climatology.md`'s diurnal-eligible list), but the ocean grid is
not currently loadable via `load_training_window`
(`core.config.grid_coordinates` raises for `"ocean"` -- its row/latitude
orientation was unverified when that guard was written; `open_questions.md`
§1.1 has since answered it empirically, but `core.config.py` has not been
updated to match). This notebook uses `was` (wind speed, wind grid, also
seasonally- and diurnally-varying) instead, so the starter code runs today;
switch to `sst` once the ocean-grid loading gap is closed.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 4, 1, tzinfo=timezone.utc)

VARIABLE = "was"
X_full = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])
print(f"loaded {X_full.dims['time']} timesteps")


## Refit over overlapping sub-windows, compare `c_lin_`

In [ ]:
from train.climatology import ClimatologyRemover

n_times = X_full.dims["time"]
window_frac = 0.6  # each sub-window covers 60% of the loaded range
window_len = int(n_times * window_frac)
starts = [0, n_times - window_len]  # earliest-possible vs. latest-possible sub-window

trend_maps = []
for start in starts:
    sub = X_full.isel(time=slice(start, start + window_len))
    remover = ClimatologyRemover(include_trend=True).fit(sub)
    trend_maps.append(remover.c_lin_[VARIABLE])
    print(f"window starting at index {start}: t_ref={remover.t_ref_.isoformat()}, "
          f"mean |c_lin|={np.nanmean(np.abs(remover.c_lin_[VARIABLE])):.5f}")

trend_diff = trend_maps[0] - trend_maps[1]
print(f"\ntrend-map difference between windows: mean abs {np.nanmean(np.abs(trend_diff)):.5f}, "
      f"vs. mean abs trend itself {np.nanmean(np.abs(trend_maps[0])):.5f}")


## Decision-rule diagnostic

If the window-to-window trend difference is comparable in magnitude to the
trend itself, that's direct, real-data evidence for `open_questions.md`
§7.1's point (c) -- two fits over different windows are not simply
noisier/cleaner estimates of one true number. Record the comparison, and
whether it should inform the (separately unresolved) training-window
decision, in `tasks/11-climatology-quasi-stationarity.md` §4.